# Streaming a HATS catalog with `LSDBStreamDataset`

`LSDBStreamDataset` feeds a HATS catalog into Hyrax's `train_stream` and `infer_stream`
verbs through [LSDB](https://docs.lsdb.io), one chunk of partitions at a time. Nothing is
ever fully materialized, so the catalog can be far larger than memory.

This notebook covers:

1. pointing the dataset at a catalog, including one you derived interactively;
2. why derived catalogs go through a registry instead of the config;
3. training on the stream, and how ragged LSDB chunks become uniform batches;
4. choosing between a finite (`"catalog"`) and an endless (`"infinite"`) stream.

In [1]:
import itertools
import tempfile

import lsdb
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

import hyrax
from hyrax.datasets import LSDBStreamDataset
from hyrax.models.model_registry import hyrax_model

## 1. Point at a catalog

The simplest case needs no code at all: set `data_location` to a HATS path or URL and
Hyrax opens it with `lsdb.open_catalog`, passing along anything in
`[data_set.LSDBStreamDataset.open_catalog_kwargs]`.

```toml
[data_request.train_stream.data]
dataset_class = "LSDBStreamDataset"
data_location = "https://data.lsdb.io/hats/gaia_dr3"
primary_id_field = "_healpix_29"
fields = ["ra", "dec", "phot_g_mean_mag"]
```

But most real work starts by *deriving* a catalog — a query, a cone search, a crossmatch.
Here we take a magnitude cut on Gaia DR3.

In [2]:
gaia = lsdb.open_catalog(
    "https://data.lsdb.io/hats/gaia_dr3",
    columns=["ra", "dec", "phot_g_mean_mag"],
)
bright = gaia.query("phot_g_mean_mag < 19")
gaia

,ra,dec,phot_g_mean_mag
npartitions=2016,,,
"Order: 2, Pixel: 0",double[pyarrow],double[pyarrow],float[pyarrow]
"Order: 2, Pixel: 1",...,...,...
...,...,...,...
"Order: 3, Pixel: 766",...,...,...
"Order: 3, Pixel: 767",...,...,...


In [2]:
def drop_nans(df):
    return df.dropna(subset="lightcurve.sap_flux").dropna(subset="lightcurve")


tess = lsdb.open_catalog(
    "https://data.lsdb.io/hats/tess/tess_lightcurve",
    columns=["ticid", "ra_obj", "dec_obj", "lightcurve"],
)

tess_filtered_nans = tess.map_partitions(drop_nans)

# from nested_pandas.utils import count_nested
# def count_points(pts):
#     # Asked to count `lc`, this will add a column called `n_lc`
#     return count_nested(pts, "lightcurve")

# tess_filtered_counted = tess_filtered_nans.map_partitions(count_points)
# tess_filtered_short = tess_filtered_counted.query("n_lightcurve < 1000")

## 2. Register a derived catalog

`bright` only exists in memory — there is no path or URL that names it. It also cannot be
dropped into the config directly: when a verb starts, Hyrax writes the whole runtime
configuration to `runtime_config.toml`, and a live `lsdb.Catalog` object is not
serializable.

So instead you register the catalog under a name and use the handle it returns as the
`data_location`. The handle is an ordinary string, so the config stays serializable.

In [3]:
data_location = LSDBStreamDataset.register_catalog("gaia_bright", bright)
data_location

'lsdb://gaia_bright'

In [3]:
data_location = LSDBStreamDataset.register_catalog("tess_filtered_nans", tess_filtered_nans)
data_location

'lsdb://tess_filtered_nans'

## 3. A small model for catalog columns

The models that ship with Hyrax expect images, so here is a compact dense autoencoder over
the three catalog columns.

The important part is `prepare_inputs`: it receives the collated batch and returns the
array the model consumes. Note it re-imports numpy and hardcodes the column names — Hyrax
writes this function out as a standalone file next to inference results, so it cannot rely
on anything defined elsewhere in the notebook.

In [4]:
@hyrax_model
class CatalogAutoencoder(nn.Module):
    """A tiny dense autoencoder over a handful of catalog columns."""

    def __init__(self, config, data_sample=None):
        super().__init__()
        self.config = config
        n_features = data_sample.shape[1]
        self.encoder = nn.Sequential(nn.Linear(n_features, 8), nn.GELU(), nn.Linear(8, 2))
        self.decoder = nn.Sequential(nn.Linear(2, 8), nn.GELU(), nn.Linear(8, n_features))

    def forward(self, batch):
        return self.encoder(batch)

    def train_batch(self, batch):
        reconstructed = self.decoder(self(batch))
        loss = F.mse_loss(reconstructed, batch)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return {"loss": loss.item()}

    def infer_batch(self, batch):
        return self(batch)

    @staticmethod
    def prepare_inputs(data_dict) -> tuple:
        """Stack the requested catalog columns into one (batch, n_features) array."""
        # Imported here because Hyrax writes this function out as a standalone file
        # alongside inference results, where notebook globals are not available.
        import numpy as np

        data = data_dict["data"]
        # Rescale each column to roughly [-1, 1]. Without this the raw ranges
        # (ra 0-360, dec -90-90, mag 3-19) make the loss diverge immediately.
        ra = np.asarray(data["ra_obj"], dtype="float32") / 180.0 - 1.0
        dec = np.asarray(data["dec_obj"], dtype="float32") / 90.0
        return np.stack([ra, dec], axis=1)
        # mag = (np.asarray(data["phot_g_mean_mag"], dtype="float32") - 12.0) / 7.0
        # return np.stack([ra, dec, mag], axis=1)

## 4. Train on the stream

`primary_id_field` is set to `_healpix_29`, the HATS spatial index. It arrives as the
DataFrame index rather than a column, and `LSDBStreamDataset` promotes it so it can be used
as the object id — useful for a derived catalog with no natural identifier of its own.

`stream_type = "infinite"` keeps batches coming indefinitely, which is what you usually
want for training. We take five batches with `islice` and stop.

In [5]:
h = hyrax.Hyrax()
h.config["model"]["name"] = "CatalogAutoencoder"
h.config["general"]["results_dir"] = tempfile.mkdtemp()
h.config["data_loader"]["batch_size"] = 4

ds_config = h.config["data_set"]["LSDBStreamDataset"]
ds_config["stream_type"] = "infinite"
ds_config["partitions_per_chunk"] = 1

h.config["data_request"] = {
    "train_stream": {
        "data": {
            "dataset_class": "LSDBStreamDataset",
            "data_location": data_location,
            "primary_id_field": "ticid",
            "fields": ["ra_obj", "dec_obj"],
        }
    }
}

with h.train_stream() as session:
    for batch, metrics in itertools.islice(session, 5):
        print(f"batch of {len(batch['object_id']):>5} objects    loss={metrics['loss']:.4f}")
        print(batch)

[2026-08-06 10:32:47,256 hyrax.models.model_registry:INFO] Setting model's self.optimizer from config: torch.optim.SGD with arguments: {'lr': 0.01, 'momentum': 0.9}.
[2026-08-06 10:32:47,257 hyrax.models.model_registry:INFO] Setting model's self.criterion from config: torch.nn.CrossEntropyLoss with default arguments.
[2026-08-06 10:32:47,257 hyrax.models.model_registry:INFO] Setting model's self.scheduler from config: torch.optim.lr_scheduler.ExponentialLR
with arguments: {'gamma': 1}.
2026-08-06 10:32:47,267 ignite.distributed.auto.auto_dataloader INFO: Use data loader kwargs for dataset '<hyrax.datasets.stre': 
	{'batch_size': None, 'collate_fn': <bound method CollationMixin.collate of <hyrax.datasets.streaming_data_provider.StreamingDataProvider object at 0x356446cf0>>, 'num_workers': 0, 'pin_memory': False}
2026/08/06 10:32:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/06 10:32:47 INFO mlflow.store.db.utils: Updating database tables
2026/08/06 10

batch of     4 objects    loss=1.0760
{'object_id': array(['235009317', '234998454', '350190353', '350274059'], dtype='<U9'), 'data': {'ra_obj': array([352.6459 , 350.76736, 354.89508, 356.63278], dtype=float32), 'dec_obj': array([-61.974384, -63.242435, -63.399345, -64.57365 ], dtype=float32)}}
batch of     4 objects    loss=1.0529
{'object_id': array(['355653322', '235043302', '235041967', '234997090'], dtype='<U9'), 'data': {'ra_obj': array([359.9791 , 354.68216, 354.4577 , 350.71844], dtype=float32), 'dec_obj': array([-65.57713 , -62.88549 , -62.36272 , -61.853188], dtype=float32)}}
batch of     4 objects    loss=1.0099
{'object_id': array(['234293493', '235010655', '267081681', '235011974'], dtype='<U9'), 'data': {'ra_obj': array([357.41437, 352.89816, 359.80307, 353.1175 ], dtype=float32), 'dec_obj': array([-62.89311 , -62.92292 , -66.37716 , -63.655006], dtype=float32)}}
batch of     4 objects    loss=0.9204
{'object_id': array(['350221443', '235004082', '235037761', '350273912'

## 5. Ragged chunks, uniform batches

LSDB hands back one DataFrame per *chunk of partitions*, and partitions vary wildly in size
— a Gaia partition holds hundreds of thousands of rows, while a crossmatched catalog might
give you nine. `LSDBStreamDataset` buffers rows across chunks and splits large ones, so
every batch is exactly `batch_size`. Only the final batch of a finite stream is ever short.

Compare what LSDB yields against what the dataset yields:

In [9]:
raw_chunk = next(iter(lsdb.streams.CatalogStream(bright, partitions_per_chunk=1, seed=0)))
print(f"one raw LSDB chunk:  {len(raw_chunk):,} rows")

# The dataset can also be driven directly, without a verb.
dataset = LSDBStreamDataset(h.config, data_location=data_location)
print("hyrax batch sizes:  ", [len(b) for b in itertools.islice(dataset, 5)])
dataset.stop()

one raw LSDB chunk:  202,682 rows
hyrax batch sizes:   [64, 64, 64, 64, 64]


## 6. `"catalog"` versus `"infinite"`

* **`stream_type = "catalog"`** wraps `lsdb.streams.CatalogStream`: a single finite pass
  that visits every object exactly once and then ends on its own. This is what you want for
  **inference**.
* **`stream_type = "infinite"`** wraps `lsdb.streams.InfiniteStream`: partitions are
  resampled forever, so batches keep arriving until you stop. This is what you want for
  **training**, since it is not bounded by one pass over the catalog.

A finite pass is easiest to see on a small local catalog. Note the last batch is short —
that is the one place a partial batch is allowed.

In [7]:
frame = pd.DataFrame(
    {
        "ra": np.linspace(0.0, 350.0, 30),
        "dec": np.linspace(-80.0, 80.0, 30),
        "phot_g_mean_mag": np.linspace(14.0, 18.0, 30),
    }
)
small = lsdb.from_dataframe(frame, ra_column="ra", dec_column="dec")
small_location = LSDBStreamDataset.register_catalog("small", small)

h2 = hyrax.Hyrax()
h2.config["model"]["name"] = "CatalogAutoencoder"
h2.config["general"]["results_dir"] = tempfile.mkdtemp()
h2.config["data_loader"]["batch_size"] = 8
h2.config["data_set"]["LSDBStreamDataset"]["stream_type"] = "catalog"
h2.config["data_set"]["LSDBStreamDataset"]["shuffle"] = False
h2.config["data_request"] = {
    "train_stream": {
        "data": {
            "dataset_class": "LSDBStreamDataset",
            "data_location": small_location,
            "primary_id_field": "_healpix_29",
            "fields": ["ra", "dec", "phot_g_mean_mag"],
        }
    }
}

seen = []
with h2.train_stream() as session:
    for batch, _metrics in session:
        seen.extend(batch["object_id"])
        print(f"batch of {len(batch['object_id'])} objects")

print(f"\nsaw {len(seen)} objects, {len(set(seen))} of them unique - the stream ended by itself")

[2026-08-05 13:22:54,960 hyrax.models.model_registry:INFO] Setting model's self.optimizer from config: torch.optim.SGD with arguments: {'lr': 0.01, 'momentum': 0.9}.


[2026-08-05 13:22:54,960 hyrax.models.model_registry:INFO] Setting model's self.criterion from config: torch.nn.CrossEntropyLoss with default arguments.


[2026-08-05 13:22:54,960 hyrax.models.model_registry:INFO] Setting model's self.scheduler from config: torch.optim.lr_scheduler.ExponentialLR
with arguments: {'gamma': 1}.


2026-08-05 13:22:54,961 ignite.distributed.auto.auto_dataloader INFO: Use data loader kwargs for dataset '<hyrax.datasets.stre': 
	{'batch_size': None, 'collate_fn': <bound method CollationMixin.collate of <hyrax.datasets.streaming_data_provider.StreamingDataProvider object at 0x74274c62d810>>, 'num_workers': 0, 'pin_memory': True}


2026/08/05 13:22:54 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


...

2026/08/05 13:22:56 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


2026/08/05 13:22:57 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...


2026/08/05 13:22:57 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


[2026-08-05 13:22:57,197 hyrax.verbs.train_stream:INFO] TrainStream session closed.


batch of 8 objects
batch of 8 objects
batch of 8 objects
batch of 6 objects

saw 30 objects, 30 of them unique - the stream ended by itself


## 7. Limitations worth knowing

**Nested columns are not supported yet.** A crossmatched catalog often carries nested
columns — a light curve or spectrum stored as a sub-table per row. Those sub-tables have
different lengths for different objects, and stacking them into a batch fails with
`setting an array element with a sequence ... inhomogeneous shape`. Request scalar or
fixed-width columns, or subclass the dataset and add a `collate_<field>` method that pads
the ragged field yourself (see
[Dataset custom collation](../notebooks/custom_dataset_collation.ipynb)).

**Stopping an infinite stream.** Prefer breaking out of the session loop, as above. Calling
`stop()` from another thread also works, but LSDB offers no way to cancel a chunk that is
already being computed, so it takes effect only at the next chunk boundary. A smaller
`partitions_per_chunk` bounds that delay.

**Cone searches.** In lsdb 0.10, passing `search_filter=lsdb.ConeSearch(...)` to
`open_catalog` produces a catalog that its own stream classes cannot iterate. Use `query()`
for cuts you want to stream, or write the filtered catalog out as HATS first.